# Advanced Visualization - UMAP

## 1) Objective
Usually, we work with high-dimensional data which makes it difficult to visualize. For that purpose, a set of tools exist that map the high-dimensional data to a lower dimensional space such that it is possible to plot the data in 2D or 3D.<br>
However, depending on the method, the projections come with certain assumptions about the data structure or the structure of the high-dimensional space. Inherently, a projection filters some information and therefore does not conserve the entire amount of information entailed in the original high-dimensional space.<br>
In this example, we like to explore these effects using a common mapping tool - UMAP - and compare the results to PCA.

## 2) Preparation

First, we import an *"Auxiliary"* package that contains a set of methods we will use for plotting and reading.

In [ ]:
import os
import sys

module_dir = os.path.abspath('../08 Codes')
sys.path.insert(0, module_dir)

In [ ]:
from Auxiliary import *

Next, we import the *MinMaxScaler* and *PCA* in order to normalize the data and for running PCA as comparison to UMAP, respecively.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

<br>

## 3) Loading Data Sets

We want to work with two data sets: The first one is a tabular data set containing features of handwritten digits:

In [ ]:
from sklearn.datasets import load_digits

This set is easy to understand, because the labels are just the digits and we therefore can immediately tell, if the UMAP projection makes sense. For example the numbers *"1"* and *"8"* should be located in well separated cluster, whereas the numbers *"3"* and *"8"* can be easly mistaken when handwritten and therefore the cluster should exhibit some overlap.

In [ ]:
digits = load_digits()

The second data set is the Alzheimer data set we know already:

In [ ]:
NamePath = FindMyFile(filename = 'AD_data.xlsx')
AD       = ReadWithAnyToolAnyMethod(filename  = NamePath, my_tool = 'pd',
                                    my_method = 'ExcelFile').parse(sheet_name = "Summary Form")

<br>

**3.1) Digits**

We extract the labels and values (X) and display the first entries

In [ ]:
XDigits = digits.data
YDigits = digits.target

In [ ]:
print(YDigits[:10])

In [ ]:
print(XDigits[:5, :])

In [ ]:
print(YDigits.shape, XDigits.shape)

Each of the 1797 digits is encoded in a 64-dimensional feature vector. As it turns out, this feature vector is just the flattened gray scale image of the handwritten digit. Let's plot the first data point as an image... 

In [ ]:
Number  = XDigits[40,:]
S       = int(len(Number)**0.5)
NumberR = Number.reshape((S, S))
plt.imshow(NumberR, cmap = 'gray_r')

...and repeat that with the first 400 entries:

In [ ]:
fig, ax_array = plt.subplots(20, 20)
axes = ax_array.flatten()
for i, ax in enumerate(axes):
    Number  = XDigits[i,:]
    NumberR = Number.reshape((S, S))
    ax.imshow(NumberR, cmap = 'gray_r')
plt.setp(axes, xticks = [], yticks = [], frame_on = False)
plt.tight_layout(h_pad = 0.5, w_pad = 0.01)

The question is, do we need all 64 dimensions? As before, we can run a PCA and take a look at the eigenvalues.<br>
Scaling first:

In [ ]:
MinMax   = MinMaxScaler()
XSDigits = MinMax.fit_transform(XDigits)

PCA:

In [ ]:
out = PCA(n_components = len(Number)).fit(XSDigits) 
eigenVec = out.components_
eigenVal = out.explained_variance_
eigenX   = out.transform(XSDigits)

Eigenvalue spectrum:

In [ ]:
xplot    = np.arange(1,len(Number)+1)

fig = plt.figure(figsize=(15, 3))
plt.bar(xplot, eigenVal, color = (0.9, 0.9, 0.9), edgecolor = 'black')
plt.xlabel('dimension')
plt.ylabel('eigenvalue')
#plt.yscale('log')
plt.xticks(xplot, fontsize = 8)
plt.show()

There seems to be no clear cut between the different eigenvalues. Nevertheless, we can plot the first two directions and see how well the cluster separate:

In [ ]:
for i in range(np.max(YDigits)+1):
    idx = np.argwhere(YDigits == i)
    plt.scatter(eigenX[idx, 0], eigenX[idx, 1], marker = '.', alpha = 0.3, label = str(i))

plt.gca().set_aspect('equal', 'datalim')
plt.title('first two PC of digits data set', fontsize = 18)
plt.legend()
plt.show()

There is some clustering, but we plotted only the first two directions which contain only

In [ ]:
var2 = np.sum(eigenVal[:2])/np.sum(eigenVal)*100

In [ ]:
print(f"{var2: 0.2f} percent of the explained variance")

Therefore, PCA is probably not the best choice.

UMAP has a completely different approach than PCA (see lecture). Instead of ignoring most of the dimensions by picking the projection to two or three directions for visualization (PCA), UMAP maps the **entire** high-dimensional to a lower, 2D or 3D space.<br>
We run UMAP with default settings (for now):

In [ ]:
newXYDigits = umap.UMAP().fit_transform(XSDigits)

And plot the result and label accordingly:

In [ ]:
for i in range(np.max(YDigits)+1):
    idx = np.argwhere(YDigits == i)
    plt.scatter(newXYDigits[idx, 0], newXYDigits[idx, 1], marker = '.', alpha = 0.3, label = str(i))

plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP projection of digits', fontsize = 18)
plt.legend()
plt.show()

We see that the different classes of digits are indeed located in relatively distinct cluster.<br>
On the other hand, UMAP needs some information about the k-nearest neighbors in order to reconstruct the original manifold and it needs an unit distance in order to estimate connectivity for constructing the graph needed for the lower-dimensional mapping. Let us vary these settings and see how the cluster change. 

In [ ]:
# changing neighbors 
for i in range(10):
    nn = i + 5
    DrawUMAP(XSDigits, YDigits, n_neighbors = nn, title = str(nn) + ' neighbors')

The larger the number of the nearest neighbors, the better the approximation of the original manifold, since we have better statistics, but at some point, we will average out the specific features of the manifold and therefore, the cluster seem to get closer to eachother with increasing number of nearest neighbors.  

Let us now vary the unit distance which is needed for calculating connectivity. The larger the distance the more data points are in reach and the more fuzzy the cluster should look.

In [ ]:
for i in range(10):
    dist = i/10
    DrawUMAP(XSDigits, YDigits, min_dist = dist, title = 'dist = ' + str(dist))

Please also note, that the orientation of the plot changes, because there is no preferred direction in space. 

<br>

**3.2) AD data set**

We want to repeat the same workflow with the Alzheimer data set now. First, we encode the features accordingly. 

In [ ]:
#extracting data
YAD       = AD['state']
XAD       = AD.drop(columns = ['state', 'p'])

In [ ]:
#feature engineering
XAD        = pd.get_dummies(XAD, columns = ['sex', 'heridity', 'marriage'], drop_first = True, dtype = 'int')

In [ ]:
#scaling
MinMax   = MinMaxScaler()
XSAD     = MinMax.fit_transform(XAD)

Let us run a PCA as with the digits data set and explore the eigenvalue spectrum

In [ ]:
#PCA
out = PCA(n_components = len(XAD.columns)).fit(XSAD) 
eigenVec = out.components_
eigenVal = out.explained_variance_
eigenX   = out.transform(XSAD)

In [ ]:
#eigenvalue spectrum
xplot    = np.arange(1,len(XAD.columns)+1)

fig = plt.figure(figsize=(15, 3))
plt.bar(xplot, eigenVal, color = (0.9, 0.9, 0.9), edgecolor = 'black')
plt.xlabel('dimension')
plt.ylabel('eigenvalue')
#plt.yscale('log')
plt.xticks(xplot)
plt.show()

In [ ]:
#visualization
idx0 = np.argwhere(YAD == 0)
idx1 = np.argwhere(YAD == 1)

plt.scatter(eigenX[idx0, 0], eigenX[idx0, 1], marker = '.', alpha = 0.3, label = 'healthy')
plt.scatter(eigenX[idx1, 0], eigenX[idx1, 1], marker = '.', alpha = 0.3, label = 'AD')
plt.gca().set_aspect('equal', 'datalim')
plt.title('first two PC of AD data set', fontsize = 18)
plt.legend()
plt.show()

We see that PCA is not very helpful here. Therefore, we generate a UMAP plot and also highlight the different states *"healthy"* and *'AD'*.  

In [ ]:
#UMAP
newXY = umap.UMAP().fit_transform(XSAD)
plt.scatter(newXY[idx0, 0], newXY[idx0, 1], alpha = 0.3, label = 'healthy')
plt.scatter(newXY[idx1, 0], newXY[idx1, 1], alpha = 0.3, label = 'AD')
plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP projection of the AD dataset', fontsize = 18)
plt.legend()
plt.show()

Unfortunately, the data points don't seem to cluster well according to their labels. I like to stress that **both, PCA and UMAP struggle with discrete features** as it is the case for this data set.  

However, we can turn the question around and label the data accoding to the feature variables and see if we find some pattern.

In [ ]:
#highlighting data according to features
for i, cols in enumerate(XAD.columns):
    plt.scatter(newXY[:, 0], newXY[:, 1], c = XSAD[:,i], cmap = 'Spectral', alpha = 0.3, marker = '.')
    plt.gca().set_aspect('equal', 'datalim')
    plt.title('UMAP projection of the AD dataset\n (' + cols + ')', fontsize = 18)
    plt.show()

Surprisingly, the data clusters with respect to many features. Among them are features which we probably would have expected like health status, but there are other features like sex and heridity which should not have an influence on the clustering. Note, however, **that the features are discrete now and these clusters are artfacts!** 